# Naive-Bayes Algorithm for Classification Task

## Group Members - Group 5

| Name                   | Email |
|:-----------------------|:------|
| Nicolas Martinez Lopez | nmartinezl@unal.edu.co|
| Anna Martina Visone    |avisone@unal.edu.co|
| Miller Barrera González|mbarrerag@unal.edu.co|
| Sergio Castro Vargas   |secastrov@unal.edu.co|

# Libraries

In [15]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

#  Load dataset

In [16]:
dataset_url = "https://raw.githubusercontent.com/nmart1nezl/DataMiningProject/refs/heads/main/datasets/classifiers_dataset.csv"
df = pd.read_csv(dataset_url)
df.drop(columns=['Unnamed: 0'], inplace=True)
df.dropna(inplace=True) # Handle missing values by dropping rows
df.head()

,category,age,industries,countryOfCitizenship,selfMade,gender,cpi_change_country,gdp_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,...,population_country,number_of_companies,finalWorth_category,is_founder,is_ceo,is_chairman,is_investor,is_entrepreneur,cpi_country_log,tax_revenue_country_country_log
0,Fashion & Retail,74.080767,Consumer Discretionary,France,False,M,1.1,2.715518e+12,65.6,102.5,...,67059887.0,75,muy alto,0,1,1,0,0,4.709981,3.226844
1,Automotive,51.767283,Consumer Discretionary,United States,True,M,7.5,2.142770e+13,88.2,101.8,...,328239523.0,7,muy alto,1,1,0,0,0,4.772716,2.360854
2,Technology,59.225188,Technology & Telecommunications,United States,True,M,7.5,2.142770e+13,88.2,101.8,...,328239523.0,100,muy alto,1,0,0,0,0,4.772716,2.360854
3,Technology,78.628337,Technology & Telecommunications,United States,True,M,7.5,2.142770e+13,88.2,101.8,...,328239523.0,5,muy alto,1,0,1,0,0,4.772716,2.360854
4,Finance & Investments,92.594114,Financials & Investments,United States,True,M,7.5,2.142770e+13,88.2,101.8,...,328239523.0,60,muy alto,0,1,1,0,0,4.772716,2.360854


In [17]:
df.shape

(2598, 22)

In [18]:
df.describe()

,age,cpi_change_country,gdp_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,life_expectancy_country,total_tax_rate_country,population_country,number_of_companies,is_founder,is_ceo,is_chairman,is_investor,is_entrepreneur,cpi_country_log,tax_revenue_country_country_log
count,2598.000000,2598.000000,2.598000e+03,2598.000000,2598.000000,2598.000000,2598.000000,2.598000e+03,2598.000000,2598.000000,2598.000000,2598.000000,2598.000000,2598.000000,2598.000000,2598.000000
mean,65.598075,4.272710,1.189355e+13,67.160778,102.822902,78.150577,43.821170,4.971908e+08,1.715935,0.658968,0.034642,0.040801,0.336798,0.581216,4.837491,2.530161
std,13.072504,3.542216,9.514697e+12,20.702226,4.597584,3.625814,11.776703,5.408432e+08,3.242857,0.474147,0.182907,0.197866,0.472706,0.493455,0.167146,0.367976
min,18.910335,-1.900000,3.154058e+09,4.000000,84.700000,54.300000,9.900000,3.801900e+04,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.610655,0.095310
25%,56.922656,1.700000,1.736426e+12,50.600000,100.200000,77.000000,36.600000,6.705989e+07,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.772716,2.360854
50%,65.718001,2.900000,1.991000e+13,66.300000,101.800000,78.500000,41.800000,3.282395e+08,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,4.772716,2.360854
75%,74.737851,7.500000,2.142770e+13,88.200000,102.600000,80.900000,59.100000,1.366418e+09,2.000000,1.000000,0.000000,0.000000,1.000000,1.000000,4.836917,2.624669
max,101.560575,53.500000,2.142770e+13,136.600000,142.100000,84.200000,106.300000,1.397715e+09,100.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5.668397,3.642836


In [19]:
df.columns

Index(['category', 'age', 'industries', 'countryOfCitizenship', 'selfMade',
       'gender', 'cpi_change_country', 'gdp_country',
       'gross_tertiary_education_enrollment',
       'gross_primary_education_enrollment_country', 'life_expectancy_country',
       'total_tax_rate_country', 'population_country', 'number_of_companies',
       'finalWorth_category', 'is_founder', 'is_ceo', 'is_chairman',
       'is_investor', 'is_entrepreneur', 'cpi_country_log',
       'tax_revenue_country_country_log'],
      dtype='object')

# Split X and Y

In [20]:
X = df.drop(columns=['selfMade', 'is_entrepreneur', 'is_founder'])
y = df['selfMade']

# Identify columns

In [21]:
categorical_cols = [
    'category',
    'countryOfCitizenship',
    'gender',
    'finalWorth_category',
    'industries' # Add industries back as a categorical feature
]

binary_cols = [
    #'is_founder',
    'is_ceo',
    'is_chairman',
    'is_investor',
    #'is_entrepreneur'
]

numeric_cols = [
    'age',
    'cpi_change_country',
    'gdp_country',
    'gross_tertiary_education_enrollment',
    'gross_primary_education_enrollment_country',
    'life_expectancy_country',
    'total_tax_rate_country',
    'population_country',
    'number_of_companies',
    'cpi_country_log',
    'tax_revenue_country_country_log'
]

# Prepare dataset

## Categorical cols

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('bin', 'passthrough', binary_cols)
    ],
    remainder='passthrough'
)

# Create pipeline

In [23]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GaussianNB())
])

# train/test split

In [24]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Split data into 70% training and 30% temporary (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,  # 30% for temporary set (validation + test)
    random_state=42,
    stratify=y
)

# Split temporary set into 50% validation and 50% test (15% each of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,  # 50% of the temporary set = 15% of total
    random_state=42,
    stratify=y_temp
)

# Encode target variable y
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

In [25]:
X_train.columns

Index(['category', 'age', 'industries', 'countryOfCitizenship', 'gender',
       'cpi_change_country', 'gdp_country',
       'gross_tertiary_education_enrollment',
       'gross_primary_education_enrollment_country', 'life_expectancy_country',
       'total_tax_rate_country', 'population_country', 'number_of_companies',
       'finalWorth_category', 'is_ceo', 'is_chairman', 'is_investor',
       'cpi_country_log', 'tax_revenue_country_country_log'],
      dtype='object')

# train

In [26]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', MinMaxScaler(),
                                                  ['age', 'cpi_change_country',
                                                   'gdp_country',
                                                   'gross_tertiary_education_enrollment',
                                                   'gross_primary_education_enrollment_country',
                                                   'life_expectancy_country',
                                                   'total_tax_rate_country',
                                                   'population_country',
                                                   'number_of_companies',
                                                   'cpi_country_log',
                                                   'tax_revenue_country_country_log']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['category',
                                                   'countryOfCitizenship',
                                                   'gender',
                                                   'finalWorth_category',
                                                   'industries']),
                                                 ('bin', 'passthrough',
                                                  ['is_ceo', 'is_chairman',
                                                   'is_investor'])])),
                ('classifier', GaussianNB())])

# metrics

In [27]:
print('--- Classification Report for Test Set ---')
y_pred = pipeline.predict(X_test)

# Inverse transform y_test and y_pred to get original labels for classification report
y_test_labels = le.inverse_transform(y_test)
y_pred_labels = le.inverse_transform(y_pred)

print(classification_report(y_test_labels, y_pred_labels))

--- Classification Report for Test Set ---
              precision    recall  f1-score   support

       False       0.33      0.96      0.49       123
        True       0.83      0.09      0.17       267

    accuracy                           0.37       390
   macro avg       0.58      0.53      0.33       390
weighted avg       0.67      0.37      0.27       390



In [28]:
print('--- Classification Report for Validation Set ---')
y_val_pred = pipeline.predict(X_val)

# Inverse transform y_val and y_val_pred to get original labels for classification report
y_val_labels = le.inverse_transform(y_val)
y_val_pred_labels = le.inverse_transform(y_val_pred)

print(classification_report(y_val_labels, y_val_pred_labels))

--- Classification Report for Validation Set ---
              precision    recall  f1-score   support

       False       0.33      0.98      0.49       122
        True       0.88      0.09      0.16       268

    accuracy                           0.36       390
   macro avg       0.61      0.53      0.32       390
weighted avg       0.71      0.36      0.26       390

